# Thử nghiệm mô hình ensemble - lần 1

Notebook này so sánh vài mô hình hồi quy đơn lẻ với một mô hình ensemble để dự đoán tuổi Abalone.

## 1. Import thư viện

In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## 2. Nạp dữ liệu và kiểm tra nhanh

In [ ]:
duong_dan_ung_vien = [
    Path('../../data/raw/abalone.csv'),
    Path('../data/raw/abalone.csv'),
    Path('data/raw/abalone.csv'),
    Path('AbaloneAge/data/raw/abalone.csv'),
]

duong_dan_du_lieu = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_du_lieu = p_day_du
        break

if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        'Khong tim thay file abalone.csv. Da thu: ' + ', '.join(str(p.resolve()) for p in duong_dan_ung_vien)
    )

df = pd.read_csv(duong_dan_du_lieu, header=None)
df.columns = [
    'sex', 'length', 'diameter', 'height',
    'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight', 'rings'
]

print('Duong dan du lieu:', duong_dan_du_lieu)
print('Kich thuoc:', df.shape)
print('So gia tri thieu:')
print(df.isnull().sum())
df.head()

## 3. Chia train, validation, test

In [ ]:
X = df.drop(columns=['rings'])
y = df['rings']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

## 4. Tiền xử lý chung

In [ ]:
cot_so = ['length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight']
cot_hang_muc = ['sex']

tien_xu_ly = ColumnTransformer(transformers=[
    ('so', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), cot_so),
    ('hang_muc', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ]), cot_hang_muc),
])

X_train_txl = tien_xu_ly.fit_transform(X_train)
X_val_txl = tien_xu_ly.transform(X_val)
X_test_txl = tien_xu_ly.transform(X_test)

ten_dac_trung = tien_xu_ly.get_feature_names_out()
print('So dac trung sau tien xu ly:', len(ten_dac_trung))

## 5. Huấn luyện các mô hình đơn lẻ

In [ ]:
mo_hinh_dict = {
    'ridge': Ridge(alpha=1.0),
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    'extra_trees': ExtraTreesRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    'gradient_boosting': GradientBoostingRegressor(random_state=42),
}

ket_qua = []
mo_hinh_da_train = {}

for ten, mo_hinh in mo_hinh_dict.items():
    mo_hinh.fit(X_train_txl, y_train)
    du_doan_val = mo_hinh.predict(X_val_txl)

    mae = mean_absolute_error(y_val, du_doan_val)
    rmse = np.sqrt(mean_squared_error(y_val, du_doan_val))
    r2 = r2_score(y_val, du_doan_val)

    ket_qua.append({
        'mo_hinh': ten,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    })
    mo_hinh_da_train[ten] = mo_hinh

bang_ket_qua = pd.DataFrame(ket_qua).sort_values(by='RMSE')
bang_ket_qua

## 6. Xây mô hình ensemble

In [ ]:
ensemble = VotingRegressor(estimators=[
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)),
    ('et', ExtraTreesRegressor(n_estimators=400, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(random_state=42)),
])

ensemble.fit(X_train_txl, y_train)
du_doan_val_ensemble = ensemble.predict(X_val_txl)

mae_ens = mean_absolute_error(y_val, du_doan_val_ensemble)
rmse_ens = np.sqrt(mean_squared_error(y_val, du_doan_val_ensemble))
r2_ens = r2_score(y_val, du_doan_val_ensemble)

bang_ensemble = pd.DataFrame([
    {'mo_hinh': 'voting_ensemble', 'MAE': mae_ens, 'RMSE': rmse_ens, 'R2': r2_ens}
])

bang_so_sanh = pd.concat([bang_ket_qua, bang_ensemble], ignore_index=True).sort_values(by='RMSE')
bang_so_sanh

## 7. Trực quan hóa so sánh

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(bang_so_sanh['mo_hinh'], bang_so_sanh['MAE'], color='steelblue')
axes[0].set_title('MAE theo mô hình')
axes[0].set_xlabel('MAE')

axes[1].barh(bang_so_sanh['mo_hinh'], bang_so_sanh['RMSE'], color='tomato')
axes[1].set_title('RMSE theo mô hình')
axes[1].set_xlabel('RMSE')

axes[2].barh(bang_so_sanh['mo_hinh'], bang_so_sanh['R2'], color='seagreen')
axes[2].set_title('R2 theo mô hình')
axes[2].set_xlabel('R2')

plt.tight_layout()
thu_muc_hinh = Path('../../outputs/figures').resolve()
thu_muc_hinh.mkdir(parents=True, exist_ok=True)
plt.savefig(thu_muc_hinh / '04_model_ensemble_try1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Da luu hinh: 04_model_ensemble_try1_comparison.png')

## 8. Đánh giá trên tập test

In [ ]:
mo_hinh_tot_nhat = bang_so_sanh.iloc[0]['mo_hinh']
print('Mo hinh tot nhat theo validation:', mo_hinh_tot_nhat)

if mo_hinh_tot_nhat == 'voting_ensemble':
    mo_hinh_cuoi = ensemble
else:
    mo_hinh_cuoi = mo_hinh_da_train[mo_hinh_tot_nhat]

du_doan_test = mo_hinh_cuoi.predict(X_test_txl)
mae_test = mean_absolute_error(y_test, du_doan_test)
rmse_test = np.sqrt(mean_squared_error(y_test, du_doan_test))
r2_test = r2_score(y_test, du_doan_test)

print(f'MAE test : {mae_test:.4f}')
print(f'RMSE test: {rmse_test:.4f}')
print(f'R2 test  : {r2_test:.4f}')

## 9. Lưu kết quả

In [ ]:
thu_muc_metrics = Path('../../outputs/metrics').resolve()
thu_muc_metrics.mkdir(parents=True, exist_ok=True)

bang_so_sanh.to_csv(thu_muc_metrics / '04_model_ensemble_try1_comparison.csv', index=False)

tong_tat = {
    'phuong_phap': 'ensemble_try1',
    'mo_hinh_tot_nhat': mo_hinh_tot_nhat,
    'validation': bang_so_sanh.iloc[0].to_dict(),
    'test': {
        'MAE': mae_test,
        'RMSE': rmse_test,
        'R2': r2_test,
    },
}

with open(thu_muc_metrics / '04_model_ensemble_try1_summary.json', 'w', encoding='utf-8') as f:
    json.dump(tong_tat, f, ensure_ascii=False, indent=2)

print('Da luu: 04_model_ensemble_try1_comparison.csv')
print('Da luu: 04_model_ensemble_try1_summary.json')